# Getting Started

## Basic Usage

The main interface is the `Benchmark` class with a context manager API:

In [ ]:
from zerobench import Benchmark

bench = Benchmark()

data = list(range(1000))
with bench(method='sum'):
    sum(data)

## Multidimensional Benchmarking

Tag your benchmarks with arbitrary keyword arguments to compare different methods and parameters:

In [ ]:
bench = Benchmark()

for n in [100, 1000, 10000]:
    data = list(range(n))
    with bench(method='sum', n=n):
        sum(data)
    with bench(method='len', n=n):
        len(data)

## Viewing Results

Display the benchmark results as a table:

In [ ]:
print(bench)

## Exporting Results

Export benchmark results to various formats:

```python
# Export to CSV
bench.write_csv('results.csv')

# Export to Parquet
bench.write_parquet('results.parquet')

# Export to Markdown
bench.write_markdown('results.md')
```

## Plotting Results

Visualize benchmark results with built-in plotting:

In [ ]:
bench.plot()

Save the plot to a file:

```python
bench.write_plot('results.pdf')
```

## Advanced Plotting with BenchmarkPlotter

For more control over plotting, use `BenchmarkPlotter` directly. You can create subplots using the `by` parameter:

In [ ]:
from zerobench import BenchmarkPlotter

# Create a plotter with subplots by method
plotter = BenchmarkPlotter(bench.to_dataframe(), by='method')
plotter.show()

### Speedup Plots with Reference

Use the `reference` parameter to add a speedup subplot comparing all methods to a baseline. The speedup is computed as `reference_time / method_time`, so values > 1 mean faster than the reference:

In [ ]:
# Reload the benchmark data
bench = Benchmark()

for n in [100, 1000, 10000]:
    data = list(range(n))
    with bench(method='sum', n=n):
        sum(data)
    with bench(method='len', n=n):
        len(data)

# Create a plotter with speedup comparison against 'sum'
plotter = BenchmarkPlotter(bench.to_dataframe(), reference='method=sum')
plotter.show()

## Configuration Options

Customize the benchmark behavior:

In [ ]:
bench = Benchmark(
    repeat=10,  # Number of measurement repetitions
    min_duration_of_repeat=0.5,  # Minimum duration per repeat (seconds)
)

data = list(range(1000))
with bench(method='sum'):
    sum(data)

## Accessing Raw Data

Get the benchmark data as a Polars DataFrame or list of dictionaries:

In [ ]:
# As Polars DataFrame
df = bench.to_dataframe()
print(df)

In [ ]:
# As list of dictionaries
reports = bench.to_dicts()
print(reports[0].keys())

## Plotting from External Data

`BenchmarkPlotter` can be used independently to visualize benchmark results stored in a CSV file:

```python
import polars as pl
from zerobench import BenchmarkPlotter

# Read benchmark results from CSV
df = pl.read_csv('benchmark_results.csv')

# Create and display the plot
plotter = BenchmarkPlotter(df, x='n', by='method')
plotter.show()

# Or save to a file
plotter.save('benchmark_plot.png')
```

The CSV file should contain at least:
- A numeric column for the x-axis (e.g., `n`, `size`)
- An `execution_times` column with list values, or a `median_execution_time` column

Example CSV structure:
```csv
n,method,execution_times,median_execution_time
100,sum,"[1.2, 1.3, 1.4]",1.3
100,len,"[0.5, 0.6, 0.7]",0.6
1000,sum,"[12.0, 12.1, 12.2]",12.1
1000,len,"[0.5, 0.6, 0.7]",0.6
```